# 🎓 UTS Machine Learning — Studi Kasus 2
**Nama:** Andreas  
**Mata Kuliah:** Machine Learning (RE603)  
**Dosen:** Naurah Nazhifah  
**Tahun Akademik:** Genap 2025–2026

---

## 📋 Latar Belakang

Adit, pemilik perusahaan XYZ, ingin membangun sistem berbasis data untuk **memprediksi gaji karyawan baru** secara objektif dan adil. Dataset historis berisi informasi usia, lama kerja, status karyawan, dan departemen.

> **Jenis Masalah:** Regresi (prediksi nilai kontinu → Gaji)  
> **Metode:** K-Nearest Neighbors (KNN) Regressor

---
## 📌 Soal 1 — Pemodelan Machine Learning (60 Poin)
### Step 1: Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error, r2_score

from IPython.display import display
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
sns.set_theme(style='whitegrid', palette='muted')
print("Library berhasil diimport!")

### Step 2: Load Dataset

In [ ]:
df = pd.read_csv('dataset_karyawan_missing.csv')
print("=" * 50)
print(f"  Jumlah Baris : {df.shape[0]}")
print(f"  Jumlah Kolom : {df.shape[1]}")
print("=" * 50)
display(df.head(10))
print("\n--- Tipe Data ---")
display(df.dtypes.to_frame(name='Tipe Data'))

### Step 3: Exploratory Data Analysis (EDA)
#### 3.1 — Cek Missing Values

In [ ]:
missing = df.isnull().sum()
pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Jumlah Missing': missing, 'Persentase (%)': pct})
display(missing_df[missing_df['Jumlah Missing'] > 0])

fig, ax = plt.subplots(figsize=(8, 4))
cols_with_na = missing[missing > 0]
bars = ax.barh(cols_with_na.index, cols_with_na.values, color='#e74c3c', edgecolor='white', height=0.5)
for bar, val in zip(bars, cols_with_na.values):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f' {val} data ({val/len(df)*100:.1f}%)', va='center', fontsize=11)
ax.set_xlabel('Jumlah Missing Value', fontsize=12)
ax.set_title('Visualisasi Missing Values per Kolom', fontsize=14, fontweight='bold')
ax.set_xlim(0, 20)
plt.tight_layout()
plt.show()

#### 3.2 — Distribusi Fitur Numerik

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
numeric_cols = ['Usia', 'Lama_Kerja', 'Gaji']
colors = ['#3498db', '#2ecc71', '#9b59b6']
labels = ['Usia (tahun)', 'Lama Kerja (tahun)', 'Gaji (Rupiah)']

for ax, col, color, label in zip(axes, numeric_cols, colors, labels):
    data = df[col].dropna()
    ax.hist(data, bins=20, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {data.mean():.0f}')
    ax.axvline(data.median(), color='orange', linestyle=':', linewidth=1.5, label=f'Median: {data.median():.0f}')
    ax.set_title(f'Distribusi {col}', fontsize=12, fontweight='bold')
    ax.set_xlabel(label)
    ax.set_ylabel('Frekuensi')
    ax.legend(fontsize=9)

plt.suptitle('Distribusi Fitur Numerik', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

#### 3.3 — Distribusi Fitur Kategorikal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

dept_counts = df['Departemen'].value_counts()
axes[0].bar(dept_counts.index, dept_counts.values,
            color=['#3498db','#e74c3c','#2ecc71','#f39c12'], edgecolor='white')
for i, (k, v) in enumerate(dept_counts.items()):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')
axes[0].set_title('Distribusi Departemen', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Departemen')
axes[0].set_ylabel('Jumlah Karyawan')

status_counts = df['Status_Karyawan'].value_counts()
wedge_props = {'edgecolor': 'white', 'linewidth': 2}
axes[1].pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%',
            colors=['#3498db', '#e74c3c'], wedgeprops=wedge_props, startangle=90,
            textprops={'fontsize': 12})
axes[1].set_title('Distribusi Status Karyawan', fontsize=12, fontweight='bold')

plt.suptitle('Distribusi Fitur Kategorikal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

#### 3.4 — Analisis Gaji berdasarkan Departemen & Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x='Departemen', y='Gaji', data=df, palette='Set2', ax=axes[0])
axes[0].set_title('Distribusi Gaji per Departemen', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Departemen')
axes[0].set_ylabel('Gaji (Rp)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'Rp {x/1e6:.1f}jt'))

sns.boxplot(x='Status_Karyawan', y='Gaji', data=df, palette='RdBu_r', ax=axes[1])
axes[1].set_title('Distribusi Gaji per Status Karyawan', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Status Karyawan')
axes[1].set_ylabel('Gaji (Rp)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'Rp {x/1e6:.1f}jt'))

plt.tight_layout()
plt.show()

#### 3.5 — Heatmap Korelasi

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
corr_df = df[['Usia', 'Lama_Kerja', 'Gaji']].dropna()
corr_matrix = corr_df.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            annot_kws={'size': 13, 'weight': 'bold'})
ax.set_title('Heatmap Korelasi Fitur Numerik', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

#### 3.6 — Scatter Plot: Gaji vs Usia & Lama Kerja

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_status = {'Tetap': '#3498db', 'Kontrak': '#e74c3c'}

for status, color in colors_status.items():
    subset = df[df['Status_Karyawan'] == status]
    axes[0].scatter(subset['Usia'], subset['Gaji'], alpha=0.6, color=color,
                    label=status, edgecolors='white', s=60)
    axes[1].scatter(subset['Lama_Kerja'], subset['Gaji'], alpha=0.6, color=color,
                    label=status, edgecolors='white', s=60)

for ax, xlabel in zip(axes, ['Usia (tahun)', 'Lama Kerja (tahun)']):
    ax.set_ylabel('Gaji (Rp)')
    ax.set_xlabel(xlabel)
    ax.legend(title='Status')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'Rp {x/1e6:.1f}jt'))

axes[0].set_title('Gaji vs Usia', fontsize=12, fontweight='bold')
axes[1].set_title('Gaji vs Lama Kerja', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📌 Soal 2 — Feature Engineering (20 Poin)

> **Feature Engineering** adalah proses transformasi dan persiapan fitur (kolom) dari data mentah menjadi format yang lebih informatif dan sesuai untuk algoritma machine learning. Tujuannya adalah meningkatkan kualitas input sehingga model dapat belajar pola dengan lebih baik.

**Contoh Feature Engineering dalam studi kasus ini:**
| Teknik | Fitur | Penjelasan |
|---|---|---|
| **Imputasi Missing Value** | `Gaji`, `Departemen` | Mengisi nilai kosong agar data lengkap sebelum diproses |
| **Label Encoding** | `Departemen`, `Status_Karyawan` | Mengubah kategori teks menjadi angka agar bisa dihitung algoritma |
| **Feature Scaling (StandardScaler)** | `Usia`, `Lama_Kerja` | Menyamakan skala fitur numerik agar KNN tidak bias pada fitur berskala besar |
| **Drop Fitur Tidak Relevan** | `ID`, `Nama` | Menghapus kolom yang tidak memberi informasi prediktif |

### Step 4: Preprocessing & Feature Engineering

In [ ]:
train = df.copy()

# 4.1 Imputasi Gaji (median per Status_Karyawan)
median_gaji_status = train.groupby('Status_Karyawan')['Gaji'].median()
print("Median Gaji per Status:")
for s, m in median_gaji_status.items():
    print(f"  {s}: Rp {m:,.0f}")

def impute_gaji(row):
    return median_gaji_status[row['Status_Karyawan']] if pd.isnull(row['Gaji']) else row['Gaji']

train['Gaji'] = train.apply(impute_gaji, axis=1)

# 4.2 Imputasi Departemen (modus)
mode_dept = train['Departemen'].mode()[0]
train['Departemen'] = train['Departemen'].fillna(mode_dept)
print(f"\nDepartemen kosong diisi dengan modus: '{mode_dept}'")

# 4.3 Drop ID dan Nama
train.drop(['ID', 'Nama'], axis=1, inplace=True)
print("\nMissing values tersisa:")
print(train.isnull().sum())

In [ ]:
# 4.4 Label Encoding — PERBAIKAN: dua encoder terpisah!
le_dept = LabelEncoder()
le_status = LabelEncoder()

train['Departemen'] = le_dept.fit_transform(train['Departemen'])
train['Status_Karyawan'] = le_status.fit_transform(train['Status_Karyawan'])

print("=== Mapping Label Encoding ===")
print("\nDepartemen:")
for i, cls in enumerate(le_dept.classes_):
    print(f"  {cls} -> {i}")
print("\nStatus_Karyawan:")
for i, cls in enumerate(le_status.classes_):
    print(f"  {cls} -> {i}")

display(train.head())

In [ ]:
# 4.5 Feature Scaling & visualisasi sebelum/sesudah
X = train.drop('Gaji', axis=1)
y = train['Gaji']

scaler = StandardScaler()
X_before = X.copy()
X_scaled = X.copy()
X_scaled[['Usia', 'Lama_Kerja']] = scaler.fit_transform(X[['Usia', 'Lama_Kerja']])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, data, title in zip(axes,
    [X_before, X_scaled],
    ['Sebelum Scaling', 'Sesudah Scaling (StandardScaler)']):
    bp = ax.boxplot([data[c] for c in ['Departemen', 'Usia', 'Lama_Kerja', 'Status_Karyawan']],
               labels=['Departemen', 'Usia', 'Lama_Kerja', 'Status'],
               patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('#3498db')
        patch.set_alpha(0.6)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Nilai')

plt.suptitle('Perbandingan Skala Fitur Sebelum & Sesudah Scaling', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

X = X_scaled.copy()

### Step 5: Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

print(f"Data Training : {X_train.shape[0]} sampel ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Data Testing  : {X_test.shape[0]} sampel ({X_test.shape[0]/len(X)*100:.0f}%)")

fig, ax = plt.subplots(figsize=(8, 2.5))
ax.barh(['Dataset'], [X_train.shape[0]], color='#3498db', label=f'Train ({X_train.shape[0]})')
ax.barh(['Dataset'], [X_test.shape[0]], left=[X_train.shape[0]], color='#e74c3c', label=f'Test ({X_test.shape[0]})')
ax.set_xlabel('Jumlah Sampel')
ax.set_title('Proporsi Train-Test Split (70:30)', fontsize=12, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### Step 6: Mencari Nilai K Optimal

In [ ]:
k_values = range(1, 21)
r2_scores, mae_scores = [], []

for k in k_values:
    knn_k = KNeighborsRegressor(n_neighbors=k)
    r2 = cross_val_score(knn_k, X_train, y_train, cv=5, scoring='r2').mean()
    mae = -cross_val_score(knn_k, X_train, y_train, cv=5, scoring='neg_mean_absolute_error').mean()
    r2_scores.append(r2)
    mae_scores.append(mae)

best_k = list(k_values)[np.argmax(r2_scores)]
print(f"Nilai K terbaik (R2): K = {best_k},  R2 = {max(r2_scores):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(k_values, r2_scores, marker='o', color='#3498db', linewidth=2, markersize=6)
axes[0].axvline(best_k, color='red', linestyle='--', label=f'K terbaik = {best_k}')
axes[0].set_title('R2 vs Nilai K', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Nilai K')
axes[0].set_ylabel('R2 Score')
axes[0].legend()

axes[1].plot(k_values, mae_scores, marker='s', color='#e74c3c', linewidth=2, markersize=6)
axes[1].axvline(best_k, color='blue', linestyle='--', label=f'K terbaik = {best_k}')
axes[1].set_title('MAE vs Nilai K', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Nilai K')
axes[1].set_ylabel('MAE (Rp)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'Rp {x:,.0f}'))
axes[1].legend()

plt.tight_layout()
plt.show()

### Step 7: Hyperparameter Tuning dengan GridSearchCV

In [ ]:
scoring = {
    'r2': make_scorer(r2_score),
    'mae': make_scorer(mean_absolute_error, greater_is_better=False),
    'rmse': make_scorer(lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)),
                        greater_is_better=False)
}

param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn = KNeighborsRegressor()
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring=scoring, refit='r2', verbose=0, n_jobs=-1)
grid_search.fit(X_train, y_train)

print("=== Parameter Terbaik ===")
print(grid_search.best_params_)

cv_results = pd.DataFrame(grid_search.cv_results_)
cv_metrics = cv_results[['params', 'mean_test_r2', 'mean_test_mae', 'mean_test_rmse']].copy()
cv_metrics['mean_test_mae'] = cv_metrics['mean_test_mae'].abs()
cv_metrics['mean_test_rmse'] = cv_metrics['mean_test_rmse'].abs()

print("\n=== Top 5 Kombinasi Parameter ===")
display(cv_metrics.sort_values(by='mean_test_r2', ascending=False).head(5).reset_index(drop=True))

#### Visualisasi Hasil GridSearch — Heatmap R²

In [ ]:
pivot_data = cv_results.pivot_table(
    values='mean_test_r2',
    index=['param_weights', 'param_metric'],
    columns='param_n_neighbors'
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot_data, annot=True, fmt='.4f', cmap='YlGnBu',
            linewidths=0.5, ax=ax, annot_kws={'size': 9})
ax.set_title('GridSearch: R2 Score per Kombinasi Parameter', fontsize=13, fontweight='bold')
ax.set_xlabel('n_neighbors')
ax.set_ylabel('weights / metric')
plt.tight_layout()
plt.show()

---
## 📌 Soal 3 — Confusion Matrix & Metrik Evaluasi (20 Poin)

> **Confusion Matrix** adalah tabel evaluasi khusus untuk masalah **klasifikasi**. Karena studi kasus ini adalah **regresi** (memprediksi nilai gaji yang kontinu), **confusion matrix tidak relevan dan tidak digunakan**.

Untuk masalah **regresi**, metrik evaluasi yang digunakan adalah:

| Metrik | Penjelasan | Interpretasi Nilai |
|---|---|---|
| **R² (R-Squared)** | Proporsi variasi data yang dapat dijelaskan model | Mendekati 1.0 = semakin baik |
| **MAE** | Rata-rata selisih absolut antara prediksi dan nilai aktual | Semakin kecil = semakin baik (dalam Rupiah) |
| **RMSE** | Seperti MAE tapi lebih sensitif terhadap error yang besar | Semakin kecil = semakin baik (dalam Rupiah) |

### Step 8: Evaluasi Model pada Test Set

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("=" * 55)
print("      HASIL EVALUASI MODEL KNN REGRESSOR")
print("=" * 55)
print(f"  R-Square (R2)  : {r2:.4f}  ({r2*100:.2f}%)")
print(f"  MAE            : Rp {mae:,.2f}")
print(f"  RMSE           : Rp {rmse:,.2f}")
print("=" * 55)
print(f"\nInterpretasi:")
print(f"  Model mampu menjelaskan {r2*100:.1f}% variasi data gaji.")
print(f"  Rata-rata kesalahan prediksi: Rp {mae:,.0f}")

#### Visualisasi Evaluasi — Actual vs Predicted, Residual, Distribusi Error

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Actual vs Predicted
axes[0].scatter(y_test, y_pred, alpha=0.6, color='#3498db', edgecolors='white', s=60)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', linewidth=2, label='Prediksi Sempurna')
axes[0].set_xlabel('Gaji Aktual (Rp)')
axes[0].set_ylabel('Gaji Prediksi (Rp)')
axes[0].set_title(f'Aktual vs Prediksi\nR2 = {r2:.4f}', fontsize=12, fontweight='bold')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}jt'))
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}jt'))
axes[0].legend()

# 2. Residual Plot
residuals = y_test.values - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.6, color='#9b59b6', edgecolors='white', s=60)
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Gaji Prediksi (Rp)')
axes[1].set_ylabel('Residual (Aktual - Prediksi)')
axes[1].set_title('Residual Plot', fontsize=12, fontweight='bold')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}jt'))

# 3. Distribusi Residual
axes[2].hist(residuals, bins=20, color='#2ecc71', edgecolor='white', alpha=0.85)
axes[2].axvline(0, color='red', linestyle='--', linewidth=2)
axes[2].axvline(np.mean(residuals), color='blue', linestyle=':', linewidth=2,
                label=f'Mean: {np.mean(residuals):,.0f}')
axes[2].set_xlabel('Residual (Rp)')
axes[2].set_ylabel('Frekuensi')
axes[2].set_title('Distribusi Residual', fontsize=12, fontweight='bold')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Perbandingan 30 sampel pertama
comparison = pd.DataFrame({
    'Aktual': y_test.values[:30],
    'Prediksi': y_pred[:30],
    'Selisih': abs(y_test.values[:30] - y_pred[:30])
})

fig, ax = plt.subplots(figsize=(15, 5))
x_idx = range(30)
ax.plot(x_idx, comparison['Aktual']/1e6, 'bo-', linewidth=1.5, markersize=6, label='Gaji Aktual')
ax.plot(x_idx, comparison['Prediksi']/1e6, 'r^--', linewidth=1.5, markersize=6, label='Gaji Prediksi')
ax.fill_between(x_idx, comparison['Aktual']/1e6, comparison['Prediksi']/1e6, alpha=0.15, color='orange')
ax.set_xlabel('Indeks Sampel (30 data pertama)')
ax.set_ylabel('Gaji (juta Rp)')
ax.set_title('Perbandingan Gaji Aktual vs Prediksi (30 Sampel)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### Step 9: Prediksi Gaji Karyawan Baru

In [ ]:
# PERBAIKAN: gunakan le_dept dan le_status yang sudah di-fit
print("=== Mapping Tersedia ===")
print("Departemen:", dict(zip(le_dept.classes_, range(len(le_dept.classes_)))))
print("Status    :", dict(zip(le_status.classes_, range(len(le_status.classes_)))))

# Input karyawan baru
nama_dept = 'HRD'
nama_status = 'Tetap'
usia_baru = 28
lama_kerja_baru = 3

new_data = pd.DataFrame({
    'Departemen': [le_dept.transform([nama_dept])[0]],
    'Usia': [usia_baru],
    'Lama_Kerja': [lama_kerja_baru],
    'Status_Karyawan': [le_status.transform([nama_status])[0]]
})

new_data[['Usia', 'Lama_Kerja']] = scaler.transform(new_data[['Usia', 'Lama_Kerja']])

prediksi = best_model.predict(new_data)[0]

print(f"\n{'='*45}")
print(f"  HASIL PREDIKSI GAJI KARYAWAN BARU")
print(f"{'='*45}")
print(f"  Departemen   : {nama_dept}")
print(f"  Status       : {nama_status}")
print(f"  Usia         : {usia_baru} tahun")
print(f"  Lama Kerja   : {lama_kerja_baru} tahun")
print(f"  {'─'*37}")
print(f"  Prediksi Gaji: Rp {prediksi:,.2f}")
print(f"{'='*45}")

---
## Kesimpulan

In [ ]:
print("=" * 60)
print("            RINGKASAN HASIL PEMODELAN")
print("=" * 60)
print(f"  Algoritma       : K-Nearest Neighbors (KNN) Regressor")
print(f"  Parameter Terbaik: {grid_search.best_params_}")
print(f"  R2 Score        : {r2:.4f} ({r2*100:.2f}%)")
print(f"  MAE             : Rp {mae:,.0f}")
print(f"  RMSE            : Rp {rmse:,.0f}")
print("=" * 60)
print(f"\nModel mampu menjelaskan {r2*100:.1f}% variasi data gaji.")
print(f"Rata-rata error prediksi sekitar Rp {mae:,.0f}.")
print("\nCatatan:")
print("  - Confusion Matrix tidak digunakan (masalah REGRESI)")
print("  - Metrik evaluasi regresi: R2, MAE, RMSE")